# AusGrid2013 hierarchy deduplication

This notebook removes duplicate hierarchy series created when a parent has exactly one child. For each group of identical rows in the summing matrix, the shallowest (parent) path is retained and its descendant path is removed.

Outputs are written beside the original AusGrid files:

- `AusGrid2013_univariate_dedup.csv`
- `AusGrid_S_dedup.csv`
- `tags_dedup.pkl`
- `AusGrid_dedup_mapping.csv`

The 279 MB long-format data file is processed in chunks to keep memory usage bounded.

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Datasets" / 'AusGrid2013'
from collections import Counter, defaultdict
from pathlib import Path
import pickle

import numpy as np
import pandas as pd


def locate_data_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd]
    candidates.extend(parent / 'Datasets' / 'AusGrid2013' for parent in (cwd, *cwd.parents))
    for candidate in candidates:
        if (candidate / str(DATA_DIR / 'AusGrid2013_univariate.csv')).is_file() and (candidate / str(DATA_DIR / 'AusGrid_S.csv')).is_file():
            return candidate
    raise FileNotFoundError('Cannot locate Datasets/AusGrid2013 with the original CSV and S matrix.')


DATA_DIR = locate_data_dir()
INPUT_Y = DATA_DIR / str(DATA_DIR / 'AusGrid2013_univariate.csv')
INPUT_S = DATA_DIR / str(DATA_DIR / 'AusGrid_S.csv')
OUTPUT_Y = DATA_DIR / str(DATA_DIR / 'AusGrid2013_univariate_dedup.csv')
OUTPUT_Y_TMP = DATA_DIR / 'AusGrid2013_univariate_dedup.tmp.csv'
OUTPUT_S = DATA_DIR / str(DATA_DIR / 'AusGrid_S_dedup.csv')
OUTPUT_TAGS = DATA_DIR / str(DATA_DIR / 'tags_dedup.pkl')
OUTPUT_MAPPING = DATA_DIR / str(DATA_DIR / 'AusGrid_dedup_mapping.csv')

CHUNK_SIZE = 500_000
VALIDATION_TIMESTAMPS = 48
AGGREGATION_ATOL = 1e-9

print(f'Data directory: {DATA_DIR}')

## 1. Detect duplicate hierarchy rows

Two hierarchy nodes describe the same series exactly when their rows in `S` are identical. In a valid tree this occurs when a parent has only one child. The shallowest path is kept.

In [ ]:
S_raw = pd.read_csv(INPUT_S)
if 'unique_id' not in S_raw.columns:
    raise KeyError(f'{INPUT_S} must contain a unique_id column.')
if S_raw['unique_id'].duplicated().any():
    raise ValueError('The original S matrix contains duplicate unique_id values.')

raw_nodes = S_raw['unique_id'].astype(str).tolist()
raw_bottoms = [str(column) for column in S_raw.columns if column != 'unique_id']
S_values = S_raw[raw_bottoms].to_numpy(dtype=np.float64)
if not np.isfinite(S_values).all():
    raise ValueError('The original S matrix contains non-finite values.')

signature_groups = defaultdict(list)
for position, row in enumerate(S_values):
    signature_groups[row.tobytes()].append(raw_nodes[position])

duplicate_groups = [group for group in signature_groups.values() if len(group) > 1]
node_position = {uid: position for position, uid in enumerate(raw_nodes)}
canonical_by_uid = {uid: uid for uid in raw_nodes}
mapping_rows = []

for group in duplicate_groups:
    keeper = min(group, key=lambda uid: (uid.count('/'), node_position[uid]))
    for uid in group:
        if uid != keeper and not uid.startswith(keeper + '/'):
            raise ValueError(
                f'Identical S rows are not a parent/descendant pair: keeper={keeper}, duplicate={uid}'
            )
        canonical_by_uid[uid] = keeper
        if uid != keeper:
            mapping_rows.append({
                'dropped_unique_id': uid,
                'kept_unique_id': keeper,
                'reason': 'identical_S_row_single_child_chain',
            })

mapping_df = pd.DataFrame(
    mapping_rows,
    columns=['dropped_unique_id', 'kept_unique_id', 'reason'],
)
display(mapping_df.head(20))
print(f'Original nodes: {len(raw_nodes)}')
print(f'Duplicate groups: {len(duplicate_groups)}')
print(f'Nodes to remove: {len(mapping_df)}')

## 2. Build the deduplicated summing matrix and tags

In [ ]:
kept_nodes = [uid for uid in raw_nodes if canonical_by_uid[uid] == uid]
dedup_bottoms = list(dict.fromkeys(canonical_by_uid.get(uid, uid) for uid in raw_bottoms))

source_columns_by_bottom = defaultdict(list)
for raw_bottom in raw_bottoms:
    source_columns_by_bottom[canonical_by_uid.get(raw_bottom, raw_bottom)].append(raw_bottom)

kept_mask = S_raw['unique_id'].astype(str).isin(kept_nodes)
dedup_bottom_data = {}
for bottom in dedup_bottoms:
    source_columns = source_columns_by_bottom[bottom]
    dedup_bottom_data[bottom] = (
        S_raw.loc[kept_mask, source_columns]
        .max(axis=1)
        .to_numpy(dtype=np.float64)
    )
S_dedup = pd.concat(
    [
        pd.DataFrame({'unique_id': S_raw.loc[kept_mask, 'unique_id'].astype(str).to_numpy()}),
        pd.DataFrame(dedup_bottom_data),
    ],
    axis=1,
)

dedup_values = S_dedup[dedup_bottoms].to_numpy(dtype=np.float64)
dedup_signatures = [row.tobytes() for row in dedup_values]
assert len(dedup_signatures) == len(set(dedup_signatures)), 'S_dedup still contains duplicate series.'
assert not S_dedup['unique_id'].duplicated().any(), 'S_dedup contains duplicate unique_id values.'
assert set(dedup_bottoms).issubset(set(kept_nodes)), 'Not every bottom series is a retained node.'
assert np.linalg.matrix_rank(dedup_values) == len(dedup_bottoms), 'S_dedup does not have full column rank.'

tags_dedup = {
    'Consumption Category': np.asarray([uid for uid in kept_nodes if uid.count('/') == 0], dtype=object),
    'Consumption Category/Postcode': np.asarray([uid for uid in kept_nodes if uid.count('/') == 1], dtype=object),
    'bottom': np.asarray(dedup_bottoms, dtype=object),
}

print(f'Deduplicated hierarchy: {len(kept_nodes)} nodes, {len(dedup_bottoms)} bottom series')
print(f'S_dedup shape: {S_dedup.shape}')
display(S_dedup.head())

## 3. Stream the long-format data and remap partition IDs

Rows belonging to removed child paths are dropped. Their retained parent rows already contain the same time series. Partition IDs are rebuilt in the same order as `S_dedup`.

In [ ]:
required_columns = ['unique_id', 'ds', 'y', 'partition_id']
label2id = {uid: position for position, uid in enumerate(kept_nodes)}
kept_node_set = set(kept_nodes)
duplicate_node_set = {uid for group in duplicate_groups for uid in group}

if OUTPUT_Y_TMP.exists():
    OUTPUT_Y_TMP.unlink()

row_counts = Counter()
duplicate_check_parts = []
sample_parts = []
validation_ds = None
wrote_header = False

for chunk_number, chunk in enumerate(
    pd.read_csv(INPUT_Y, usecols=required_columns, chunksize=CHUNK_SIZE),
    start=1,
):
    chunk['unique_id'] = chunk['unique_id'].astype(str)
    if validation_ds is None:
        validation_ds = chunk['ds'].drop_duplicates().head(VALIDATION_TIMESTAMPS).tolist()

    duplicate_part = chunk.loc[
        chunk['unique_id'].isin(duplicate_node_set),
        ['unique_id', 'ds', 'y'],
    ]
    if not duplicate_part.empty:
        duplicate_check_parts.append(duplicate_part.copy())

    out = chunk.loc[chunk['unique_id'].isin(kept_node_set), ['unique_id', 'ds', 'y']].copy()
    out['partition_id'] = out['unique_id'].map(label2id)
    if out['partition_id'].isna().any():
        raise AssertionError('A retained node could not be mapped to a partition_id.')
    out['partition_id'] = out['partition_id'].astype(np.int64)
    row_counts.update(out['unique_id'].value_counts().to_dict())

    sample = out.loc[out['ds'].isin(validation_ds), ['unique_id', 'ds', 'y']]
    if not sample.empty:
        sample_parts.append(sample.copy())

    out.to_csv(OUTPUT_Y_TMP, mode='a', header=not wrote_header, index=False)
    wrote_header = True
    print(f'Processed chunk {chunk_number}: retained {len(out):,} rows')

assert wrote_header, 'No rows were written to the deduplicated data file.'
assert set(row_counts) == kept_node_set, 'Some retained hierarchy nodes have no data rows.'
print(f'Temporary deduplicated rows: {sum(row_counts.values()):,}')

## 4. Validate duplicate equality and hierarchical aggregation

In [ ]:
duplicate_values = pd.concat(duplicate_check_parts, ignore_index=True)
if duplicate_values.duplicated(['unique_id', 'ds']).any():
    raise AssertionError('The original data contains duplicate (unique_id, ds) rows.')

max_duplicate_error = 0.0
for group in duplicate_groups:
    keeper = min(group, key=lambda uid: (uid.count('/'), node_position[uid]))
    group_wide = (
        duplicate_values.loc[duplicate_values['unique_id'].isin(group)]
        .pivot(index='ds', columns='unique_id', values='y')
    )
    if set(group) != set(group_wide.columns):
        raise AssertionError(f'Missing time-series data for duplicate group {group}.')
    for duplicate_uid in group:
        if duplicate_uid == keeper:
            continue
        error = np.nanmax(np.abs(group_wide[keeper].to_numpy() - group_wide[duplicate_uid].to_numpy()))
        max_duplicate_error = max(max_duplicate_error, float(error))

sample_long = pd.concat(sample_parts, ignore_index=True)
if sample_long.duplicated(['unique_id', 'ds']).any():
    raise AssertionError('Validation sample contains duplicate (unique_id, ds) rows.')
sample_wide = sample_long.pivot(index='ds', columns='unique_id', values='y').sort_index()
missing_sample_nodes = set(kept_nodes) - set(sample_wide.columns)
if missing_sample_nodes:
    raise AssertionError(f'Validation sample is missing nodes: {sorted(missing_sample_nodes)}')

reconstructed = sample_wide[dedup_bottoms].to_numpy(dtype=np.float64) @ dedup_values.T
target = sample_wide[kept_nodes].to_numpy(dtype=np.float64)
max_aggregation_error = float(np.nanmax(np.abs(reconstructed - target)))

print(f'Max parent/child duplicate error: {max_duplicate_error:.3e}')
print(f'Max sampled aggregation error: {max_aggregation_error:.3e}')
assert max_duplicate_error <= AGGREGATION_ATOL, 'A parent and its single child do not contain the same values.'
assert max_aggregation_error <= AGGREGATION_ATOL, 'S_dedup cannot reconstruct the retained hierarchy.'

## 5. Publish validated outputs

In [ ]:
OUTPUT_Y_TMP.replace(OUTPUT_Y)
S_dedup.to_csv(OUTPUT_S, index=False)
mapping_df.to_csv(OUTPUT_MAPPING, index=False)
with OUTPUT_TAGS.open('wb') as file:
    pickle.dump(tags_dedup, file)

print('Saved validated outputs:')
for path in [OUTPUT_Y, OUTPUT_S, OUTPUT_TAGS, OUTPUT_MAPPING]:
    print(f'  {path}')
print(f'Final hierarchy: {len(kept_nodes)} nodes, {len(dedup_bottoms)} bottom series')